# Error debugging

We ran into the following error:

`Traceback (most recent call last):
  File "/aloy/home/ddalton/projects/scGPT_playground/scripts/cli/scGPT-ft.py", line 1834, in <module>
    ) = test_2(best_model, valid_adata)
  File "/aloy/home/ddalton/projects/scGPT_playground/scripts/cli/scGPT-ft.py", line 698, in test_2
    auroc = roc_auc_score(disease_multilabels, predictions, average="macro")
  File "/opt/conda/lib/python3.10/site-packages/sklearn/utils/_param_validation.py", line 213, in wrapper
    return func(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_ranking.py", line 648, in roc_auc_score
    return _average_binary_score(
  File "/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_base.py", line 119, in _average_binary_score
    score[c] = binary_metric(y_true_c, y_score_c, sample_weight=score_weight)
  File "/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_ranking.py", line 382, in _binary_roc_auc_score
    raise ValueError(
ValueError: Only one class present in y_true. ROC AUC score is not defined in that case.
Traceback (most recent call last):
  File "/aloy/home/ddalton/projects/scGPT_playground/scripts/cli/scGPT-ft.py", line 1834, in <module>
    ) = test_2(best_model, valid_adata)
  File "/aloy/home/ddalton/projects/scGPT_playground/scripts/cli/scGPT-ft.py", line 698, in test_2
    auroc = roc_auc_score(disease_multilabels, predictions, average="macro")
  File "/opt/conda/lib/python3.10/site-packages/sklearn/utils/_param_validation.py", line 213, in wrapper
    return func(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_ranking.py", line 648, in roc_auc_score
    return _average_binary_score(
  File "/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_base.py", line 119, in _average_binary_score
    score[c] = binary_metric(y_true_c, y_score_c, sample_weight=score_weight)
  File "/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_ranking.py", line 382, in _binary_roc_auc_score
    raise ValueError(
ValueError: Only one class present in y_true. ROC AUC score is not defined in that case.`


Which means we have a disease label with only 0s or 1s - which should never happen

In [1]:
import obonet
import sys
sys.path.append("../../")
from src.utils import utils as u
import scanpy as sc

# load adata
adata = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-14-01/data.h5ad")


# get Disease Ontology graph
do_g = u.load_do_graph()

# get sanchez IC
doid_2_ic = u.get_sanchez_ic(do_g)

#! IMPORTANT - FIX CONTROL CLASS


#! REMOVE - THIS IS A QUICK AND UGLY FIX
only_control = True
if only_control:
    adata = adata[adata.obs["doid_id"] != "Control"].copy()


Number of DO leaves: 9018


In [2]:


# get nodes
# _class_nodes = u.get_lvl1_nodes(do_g)
_class_nodes = u.get_n_lowest_ic_nodes(doid_2_ic, 50)
# benchmark class nodes - use the sames as in the single classifier benchmark
# _class_nodes = adata.obs["do_id"].unique().tolist()

# Generate multilabel vectors for level 1 nodes
Y_multilabel, _class_nodes = u.generate_multilabel_vectors(adata, do_g, _class_nodes)
print(f"Generated multilabel vectors for level 1 nodes with shape {Y_multilabel.shape}")

# Clean multilabel vectors by removing nodes with no samples
Y_multilabel, _class_nodes = u.clean_multilabel_vectors(Y_multilabel, _class_nodes)
print(f"Cleaned multilabel vectors for top 50 nodes with shape {Y_multilabel.shape}")

# Check the multilabel vector for level 1 nodes
u.check_multilabel_vector(Y_multilabel, _class_nodes, do_g)

# get doids and class names
Y_multilabel_doid, Y_multilabel_name = u.get_multilabel_data(Y_multilabel, _class_nodes, do_g)


Generated multilabel vectors for level 1 nodes with shape (21246, 50)
Cleaned multilabel vectors for top 50 nodes with shape (21246, 47)
554 samples	Node DOID:0014667 - disease of metabolism
518 samples	Node DOID:0050117 - disease by infectious agent
214 samples	Node DOID:0050155 - sensory system disease
422 samples	Node DOID:0050177 - monogenic disease
8477 samples	Node DOID:0050686 - organ system cancer
8103 samples	Node DOID:0050687 - cell type cancer
18 samples	Node DOID:0050735 - X-linked monogenic disease
201 samples	Node DOID:0050736 - autosomal dominant disease
158 samples	Node DOID:0050737 - autosomal recessive disease
359 samples	Node DOID:0050739 - autosomal genetic disease
9 samples	Node DOID:0060072 - benign neoplasm
9 samples	Node DOID:0060085 - organ system benign neoplasm
162 samples	Node DOID:0080000 - muscular disease
653 samples	Node DOID:0080001 - bone disease
27 samples	Node DOID:0080015 - physical disorder
352 samples	Node DOID:114 - heart disease
617 samples	Node

In [3]:
import numpy as np
max(np.sum(Y_multilabel, axis=1))

np.int64(9)

In [4]:
from sklearn.metrics import roc_auc_score


y_true = np.array([[1,1,1,0],[1,0,1,0], [1,1,0,0]])
y_test = np.array([[1,1,1,0],[1,0,1,0], [1,1,0,0]])

roc_auc_score(y_true, y_test, average='micro', multi_class='ovr')

1.0

In [5]:
from sklearn.model_selection import train_test_split

celltypes_labels = np.array([0, 1, 0, 1, 0, 1, 2, 1, 1 ,1, 1, 0, 0, 0, 2, 0, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 0, 0, 0, 0])

t_idxs, v_idxs = train_test_split(
        np.arange(len(celltypes_labels)), test_size=0.1, shuffle=True, stratify=celltypes_labels
    )

celltypes_labels[v_idxs]

ValueError: The test_size = 3 should be greater or equal to the number of classes = 4

In [ ]:
adata.obs

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease
4,DSA00004.GSM7009978.Case,GSE224022,GSE224022,3,3,DSA00004,Retina,19402,Retinoblastoma,Retinoblastoma,Retinoblastoma,RNA-Seq,D,DOID:768,DOID:768,retinoblastoma
5,DSA00004.GSM7009979.Case,GSE224022,GSE224022,3,3,DSA00004,Retina,19402,Retinoblastoma,Retinoblastoma,Retinoblastoma,RNA-Seq,D,DOID:768,DOID:768,retinoblastoma
6,DSA00004.GSM7009980.Case,GSE224022,GSE224022,3,3,DSA00004,Retina,19402,Retinoblastoma,Retinoblastoma,Retinoblastoma,RNA-Seq,D,DOID:768,DOID:768,retinoblastoma
7,DSA00004.GSM7009981.Case,GSE224022,GSE224022,3,3,DSA00004,Retina,19402,Retinoblastoma,Retinoblastoma,Retinoblastoma,RNA-Seq,D,DOID:768,DOID:768,retinoblastoma
8,DSA00004.GSM7009982.Case,GSE224022,GSE224022,3,3,DSA00004,Retina,19402,Retinoblastoma,Retinoblastoma,Retinoblastoma,RNA-Seq,D,DOID:768,DOID:768,retinoblastoma
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35621,DSA10149.GSM5008706.Case,GSE164376,GSE164376,316,316,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,D,DOID:5419,DOID:5419,schizophrenia
35622,DSA10149.GSM5008707.Case,GSE164376,GSE164376,316,316,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,D,DOID:5419,DOID:5419,schizophrenia
35623,DSA10149.GSM5008708.Case,GSE164376,GSE164376,316,316,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,D,DOID:5419,DOID:5419,schizophrenia
35624,DSA10149.GSM5008709.Case,GSE164376,GSE164376,316,316,DSA10149,nan,19402,Schizophrenia,Schizophrenia,Schizophrenia,RNA-Seq,D,DOID:5419,DOID:5419,schizophrenia


In [ ]:
train_test_split(
       [0, 1, 0, 1, 0, 1, 2, 1, 1 ,1, 1, 0, 0, 0, 2, 0, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 0, 0, 0, 0], train_size=0.9, shuffle=True, stratify=[0, 1, 0, 1, 0, 1, 2, 1, 1 ,1, 1, 0, 0, 0, 2, 0, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 0, 0, 0, 0]
    )

ValueError: The test_size = 3 should be greater or equal to the number of classes = 4